<a href="https://colab.research.google.com/github/dennzii/Low-Memory-Difusion-Based-Virtual-Try-On-via-Quantized-Stable-Difusion-Backbones/blob/main/concat_selfattn_sd1_5_lora_finetune6_04_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms
import torch
from diffusers import DiffusionPipeline
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("marquis03/high-resolution-viton-zalando-dataset")

print("Path to dataset files:", path)

In [ ]:
import os
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T


class VITONDataset(Dataset):

    def __init__(self, dataset_path, image_size=(512, 512)):
        self.dataset_path = dataset_path
        self.image_size = image_size

        self.file_names = sorted(os.listdir(os.path.join(dataset_path, "image")))

        self.to_tensor = T.ToTensor()
        self.resize = T.Resize(image_size, interpolation=T.InterpolationMode.BILINEAR)

    def __len__(self):
        return len(self.file_names)

    def load_rgb_tensor(self, path):
        img = Image.open(path).convert("RGB")
        img = self.resize(img)
        return self.to_tensor(img)  # (3,H,W)

    def concat_images_tensor(self, cloth, image):
        # cloth + image → width concat (same as PIL paste)
        return torch.cat([cloth, image], dim=2)  # (3, H, 2W)

    def zero_pad_left_tensor(self, mask):
        # mask: (1,H,W) → (1,H,2W)
        B, H, W = mask.shape
        pad = torch.zeros((1, H, W), device=mask.device)
        return torch.cat([pad, mask], dim=2)

    def extract_mask_from_agnostic(self, agnostic_tensor):

      r, g, b = agnostic_tensor[0], agnostic_tensor[1], agnostic_tensor[2]

      mask = (
          (torch.abs(r - g) < 0.02) &
          (torch.abs(r - b) < 0.02) &
          (r > 0.45) & (r < 0.75)
      ).float().unsqueeze(0)  # (1,H,W)

      mask = mask.unsqueeze(0)  # (1,1,H,W)

      #erosion
      mask = -F.max_pool2d(-mask, kernel_size=3, stride=1, padding=1)

      #dilation
      mask = F.max_pool2d(mask, kernel_size=3, stride=1, padding=1)

      # squeeze back
      mask = mask.squeeze(0)

      return mask  # (1,H,W)

    def __getitem__(self, idx):

        file_name = self.file_names[idx]

        cloth_path = os.path.join(self.dataset_path, "cloth", file_name)
        image_path = os.path.join(self.dataset_path, "image", file_name)
        agnostic_path = os.path.join(self.dataset_path, "agnostic-v3.2", file_name)

        # -------------------------
        # LOAD (tensor directly)
        # -------------------------
        cloth = self.load_rgb_tensor(cloth_path)
        image = self.load_rgb_tensor(image_path)
        agnostic = self.load_rgb_tensor(agnostic_path)

        # -------------------------
        # CONCAT (same logic)
        # -------------------------
        concat_tensor = self.concat_images_tensor(cloth, image)

        # -------------------------
        # MASK (same logic)
        # -------------------------
        mask_512 = self.extract_mask_from_agnostic(agnostic)

        mask_pad = self.zero_pad_left_tensor(mask_512)

        mask = (mask_pad > 0.5).float()

        return concat_tensor, mask

In [ ]:
trans = transforms.Compose([transforms.ToTensor()])

train_ds = VITONDataset(dataset_path=path+"/train")



In [ ]:
train_loader = DataLoader(train_ds,batch_size=8,shuffle=True,num_workers=4,
    pin_memory=True,
    persistent_workers=True)


In [ ]:
from google.colab import drive
import os


save_dir = "/content/drive/MyDrive/VTON_Project/checkpoints"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
import torch.nn as nn
from diffusers import UNet2DConditionModel

unet = UNet2DConditionModel.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-inpainting",
    subfolder="unet"
).to(device)

# önce HER ŞEYİ freeze et
for param in unet.parameters():
    param.requires_grad = False

# sadece attn1 layerlarını aç
for name, module in unet.named_modules():
    if "attn1" in name:
        for param in module.parameters():
            param.requires_grad = True

# kontrol
trainable = sum(p.numel() for p in unet.parameters() if p.requires_grad)
print("Trainable params:", trainable)

In [ ]:
from diffusers.models.autoencoders import AutoencoderKL
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
vae = AutoencoderKL.from_pretrained("stable-diffusion-v1-5/stable-diffusion-inpainting", subfolder="vae").to(device)
vae.eval()

In [ ]:
noise_scheduler = DDPMScheduler.from_pretrained("stable-diffusion-v1-5/stable-diffusion-inpainting", subfolder="scheduler")

attn1 self attn
attn2 cross attn

input text embed = torch.zeros olcak. etkisini yok edicem.

üstüne lora yapıcam.

In [ ]:
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, unet.parameters()),
    lr=1e-5
)

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

epochs = 10
steps_per_epoch = len(train_loader)

lr_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=steps_per_epoch * epochs,
    eta_min=1e-6
)

In [ ]:
from tqdm import tqdm
import torch.nn.functional as F
import shutil # Import shutil for file operations

scaler = torch.amp.GradScaler('cuda') # Updated GradScaler

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
unet.train()

best_loss = float("inf")

for epoch in range(epochs):


    epoch_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in pbar:


        images = batch[0].to(device)
        masks = batch[1].to(device)

        batch_size = images.shape[0]


        images = images * 2.0 - 1 #tensoru -1-1 arasına çekiyoruz SD VAE istediği aralık.
        masked_images = images * (1-masks)


        with torch.no_grad():
            latents = vae.encode(images).latent_dist.sample()
            latents = latents * vae.config.scaling_factor

            masked_latents = vae.encode(masked_images).latent_dist.sample()
            masked_latents = masked_latents * vae.config.scaling_factor


        masks = F.interpolate(masks,size = latents.shape[-2:],mode="nearest")#(B, C, H, W) dolayısıyla H,W



        noise = torch.randn_like(latents)#Noise sample ediyoruz gaussian distrubutiondan.
        timesteps = torch.randint(
            0,
            noise_scheduler.num_train_timesteps,
            (latents.shape[0],),
            device=device
        ).long()

        noisy_latents = noise_scheduler.add_noise(latents,noise,timesteps)

        model_input = torch.cat(tensors=
                                  [noisy_latents,
                                    masks,
                                    masked_latents],dim=1) #(B, C, H, W) dim 1 = Channel-wise concat

        encoder_hidden_states = torch.zeros(size = (latents.shape[0],77,768),device=device) # batchsize kadarlık bir zero embedding oluşturuyoruz.

        with torch.amp.autocast('cuda'): # Updated autocast

          noise_pred = unet(model_input,
                            timesteps,
                            encoder_hidden_states= encoder_hidden_states).sample

          loss = F.mse_loss(noise_pred,noise)



        optimizer.zero_grad()

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        pbar.set_postfix({"loss": loss.item()})
        lr_scheduler.step()

    avg_loss = epoch_loss / len(train_loader)
    print(f"\nEpoch {epoch+1} Ortalama Loss: {avg_loss:.6f}")



    temp_save_path = os.path.join("/content/", f"checkpoint_epoch_{epoch}_pbc_fp32_{avg_loss:.6f}.pt")

    torch.save(
        {
            "epoch": epoch,
            "unet": unet.state_dict(),
            "optimizer": optimizer.state_dict(),
            "loss": avg_loss,
        },
        temp_save_path
    )
    # Copy the checkpoint to the final save_dir
    shutil.copy(temp_save_path, os.path.join(save_dir, f"checkpoint_epoch_{epoch}_pbc_fp32_{avg_loss:.6f}.pt"))


In [ ]:
test_ds = VITONDataset(dataset_path=path+"/test")
test_loader = DataLoader(test_ds,batch_size = 8,shuffle=False)


In [ ]:


files = os.listdir(save_dir)

filtered = []

for f in files:
    if (
        f.startswith("checkpoint_epoch_") and
        f.endswith(".pt") and
        "_pbc_fp32_" in f
    ):
      filtered.append(f)

In [ ]:
filtered

In [ ]:
import torch
import torch.nn.functional as F
from torchvision.utils import save_image
import os

model_list = []

device = "cuda"

dir = "/content/"
os.makedirs(save_dir, exist_ok=True)

# -------------------------
# LOAD MODEL
# -------------------------



ckpt = torch.load("/content/checkpoint_epoch_last_pbc.pt", map_location=device)

unet.load_state_dict(ckpt["unet"])
unet.eval().to(device)

vae.eval().to(device)

# -------------------------
# INFERENCE LOOP
# -------------------------
with torch.no_grad():

    for i, batch in enumerate(test_loader):

        concat_tensor, mask_tensor = batch

        concat_tensor = concat_tensor.to(device)
        mask_tensor   = mask_tensor.to(device)

        # -------------------------
        # SAMPLE AL
        # -------------------------
        image = concat_tensor[0]   # [C,H,W]
        mask  = mask_tensor[0]     # [C,H,W] ya da [1,H,W]

        # mask tek kanal olsun
        if mask.dim() == 3 and mask.shape[0] != 1:
            mask = mask[:1]

        # -------------------------
        # PREPROCESS
        # -------------------------
        image = image * 2.0 - 1.0
        mask  = (mask > 0.5).float()

        # batch dim
        image = image.unsqueeze(0)   # [1,C,H,W]
        mask  = mask.unsqueeze(0)    # [1,1,H,W]

        # -------------------------
        # VAE ENCODE
        # -------------------------
        image_latents = vae.encode(image.half()).latent_dist.sample()
        image_latents = image_latents * vae.config.scaling_factor

        masked_image = image * (1 - mask)

        masked_latents = vae.encode(masked_image.half()).latent_dist.sample()
        masked_latents = masked_latents * vae.config.scaling_factor

        # mask → latent size
        mask = F.interpolate(
            mask,
            size=image_latents.shape[-2:],
            mode="nearest"
        )

        # -------------------------
        # NOISE + TIMESTEPS
        # -------------------------
        num_steps = 50
        strength = 1.0

        noise_scheduler.set_timesteps(num_steps, device=device)

        init_timestep = int(num_steps * strength)
        t_start = max(num_steps - init_timestep, 0)

        timesteps = noise_scheduler.timesteps[t_start:]

        noise = torch.randn_like(image_latents)

        latents = noise_scheduler.add_noise(
            image_latents,
            noise,
            timesteps[0]
        )

        # -------------------------
        # DENOISING LOOP
        # -------------------------
        for t in timesteps:

            model_input = torch.cat(
                [latents, mask, masked_latents],
                dim=1
            )

            encoder_hidden_states = torch.zeros(
                (1, 77, 768),
                device=device
            )

            noise_pred = unet(
                model_input,
                t,
                encoder_hidden_states=encoder_hidden_states
            ).sample

            latents = noise_scheduler.step(
                noise_pred,
                t,
                latents
            ).prev_sample

        # -------------------------
        # DECODE
        # -------------------------
        latents = latents / vae.config.scaling_factor

        output = vae.decode(latents.half()).sample

        output = (output / 2 + 0.5).clamp(0, 1)

        # -------------------------
        # SAVE OUTPUT
        # -------------------------
        save_image(output,
                  os.path.join(dir, f"{i}_output.png"))

        print(f"Saved sample {i}")

        if i == 10:
            break

In [ ]:
!pip install lpips

In [ ]:
device = "cuda"

lpips_fn = lpips.LPIPS(net="alex").to(device)
lpips_fn.eval()

In [ ]:
import os
import torch
import torch.nn.functional as F
import lpips
from tqdm import tqdm



save_dir = "/content/"
os.makedirs(save_dir, exist_ok=True)

def to_lpips_range(x):
    return x * 2.0 - 1.0

@torch.no_grad()
def inference_one_model_lpips(model_path, title, save_images=False, max_batches=None):
    ckpt = torch.load(model_path, map_location=device)

    unet.load_state_dict(ckpt["unet"])
    unet.eval().to(device)
    vae.eval().to(device)

    sample_scores = []
    total_sum = 0.0
    total_count = 0

    for batch_idx, batch in enumerate(tqdm(test_loader, desc=title)):
        if batch_idx >= 50:
          break

        concat_tensor, mask_tensor = batch
        concat_tensor = concat_tensor.to(device)   # [B,3,H,2W]
        mask_tensor = mask_tensor.to(device)       # [B,1,H,2W] veya [B,3,H,2W]

        image = concat_tensor                       # tüm batch
        mask = mask_tensor

        if mask.dim() == 4 and mask.shape[1] != 1:
            mask = mask[:, :1]

        image = image * 2.0 - 1.0
        mask = (mask > 0.5).float()

        # masked image
        masked_image = image * (1 - mask)

        # VAE encode
        image_latents = vae.encode(image).latent_dist.sample()
        image_latents = image_latents * vae.config.scaling_factor

        masked_latents = vae.encode(masked_image).latent_dist.sample()
        masked_latents = masked_latents * vae.config.scaling_factor

        # mask latent size
        mask_latent = F.interpolate(
            mask,
            size=image_latents.shape[-2:],
            mode="nearest"
        )

        # noise + timesteps
        num_steps = 25
        strength = 1.0

        noise_scheduler.set_timesteps(num_steps, device=device)

        init_timestep = int(num_steps * strength)
        t_start = max(num_steps - init_timestep, 0)
        timesteps = noise_scheduler.timesteps[t_start:]

        noise = torch.randn_like(image_latents)
        latents = noise_scheduler.add_noise(
            image_latents,
            noise,
            timesteps[0]
        )

        # denoising
        for t in timesteps:
            model_input = torch.cat(
                [latents, mask_latent, masked_latents],
                dim=1
            )

            encoder_hidden_states = torch.zeros(
                (image.shape[0], 77, 768),
                device=device
            )

            noise_pred = unet(
                model_input,
                t,
                encoder_hidden_states=encoder_hidden_states
            ).sample

            latents = noise_scheduler.step(
                noise_pred,
                t,
                latents
            ).prev_sample

        # decode
        latents = latents / vae.config.scaling_factor
        output = vae.decode(latents).sample
        output = (output / 2 + 0.5).clamp(0, 1)   # [B,3,H,2W]

        # right half only
        pred_right = output[:, :, :, 512:]
        ref_right = concat_tensor[:, :, :, 512:]

        # sadece synth edilen kısmı modelden, geri kalanı GT'den al
        mask_right = mask[:, :, :, 512:]
        pred_right = pred_right * mask_right + ref_right * (1 - mask_right)

        # LPIPS range
        pred_lpips = to_lpips_range(pred_right)
        ref_lpips = to_lpips_range(ref_right)

        # batch-wise LPIPS
        scores = lpips_fn(pred_lpips, ref_lpips)   # [B,1,1,1]
        scores = scores.view(scores.shape[0])

        total_sum += scores.sum().item()
        total_count += scores.numel()
        sample_scores.extend(scores.detach().cpu().tolist())

        if save_images:
            from torchvision.utils import save_image
            for b in range(output.shape[0]):
                save_image(output[b], os.path.join(save_dir, f"{title}_b{batch_idx}_s{b}_output.png"))
                save_image(pred_right[b], os.path.join(save_dir, f"{title}_b{batch_idx}_s{b}_pred_right.png"))
                save_image(ref_right[b], os.path.join(save_dir, f"{title}_b{batch_idx}_s{b}_ref_right.png"))

    avg_score = total_sum / total_count if total_count > 0 else float("nan")
    return avg_score, sample_scores




model_name = "/content/drive/MyDrive/VTON_Project/checkpoints/checkpoint_epoch_4_pbc_fp32_0.020960.pt"

model_path = os.path.join(save_dir,model_name)

avg_lpips, per_sample = inference_one_model_lpips(model_path, title="whole", save_images=False)


print(f"avg :{avg_lpips}")

In [ ]:
a = total_sum / total_count

print(a)

In [ ]:


avg_score, sample_scores = inference_one_model_lpips(model_path,"wholetest",True)

print(f"avg: {avg_score}")

print(sample_scores)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import cv2
import os
import random

device = "cuda"

save_path = "/content/denoise_video.mp4"

# -------------------------
# RANDOM SAMPLE SEÇ
# -------------------------
dataset = test_loader.dataset
idx = random.randint(0, len(dataset) - 1)
concat_tensor, mask_tensor = dataset[idx]

image = concat_tensor.to(device)   # [C,H,W]
mask  = mask_tensor.to(device)

if mask.dim() == 3 and mask.shape[0] != 1:
    mask = mask[:1]

# preprocess
image = image * 2.0 - 1.0
mask  = (mask > 0.5).float()

image = image.unsqueeze(0)
mask  = mask.unsqueeze(0)

# -------------------------
# VIDEO SETUP
# -------------------------
fps = 10
H, W = image.shape[-2:]
frame_size = (W * 2, H)  # sol + sağ

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
video = cv2.VideoWriter(save_path, fourcc, fps, frame_size)

# -------------------------
# SOL: MASKED IMAGE (FULL)
# -------------------------
masked_image = image * (1 - mask)

left_vis = (masked_image / 2 + 0.5).clamp(0,1)
left_np = left_vis.cpu().permute(0,2,3,1).numpy()[0]
left_np = (left_np * 255).astype(np.uint8)

# -------------------------
# VAE ENCODE
# -------------------------
with torch.no_grad():

    image_latents = vae.encode(image).latent_dist.sample()
    image_latents *= vae.config.scaling_factor

    masked_latents = vae.encode(masked_image).latent_dist.sample()
    masked_latents *= vae.config.scaling_factor

    mask_latent = F.interpolate(mask, size=image_latents.shape[-2:], mode="nearest")

    # -------------------------
    # NOISE INIT
    # -------------------------
    num_steps = 50
    noise_scheduler.set_timesteps(num_steps, device=device)

    noise = torch.randn_like(image_latents)

    latents = noise_scheduler.add_noise(
        image_latents,
        noise,
        noise_scheduler.timesteps[0]
    )

    # -------------------------
    # DENOISING
    # -------------------------
    for step, t in enumerate(noise_scheduler.timesteps):

        model_input = torch.cat(
            [latents, mask_latent, masked_latents],
            dim=1
        )

        encoder_hidden_states = torch.zeros((1,77,768), device=device)

        noise_pred = unet(
            model_input,
            t,
            encoder_hidden_states=encoder_hidden_states
        ).sample

        latents = noise_scheduler.step(
            noise_pred,
            t,
            latents
        ).prev_sample

        # 🔥 daha smooth video için her 2 step
        if step % 2 != 0:
            continue

        # decode
        temp_latents = latents / vae.config.scaling_factor
        img = vae.decode(temp_latents).sample
        img = (img / 2 + 0.5).clamp(0,1)

        right_np = img.cpu().permute(0,2,3,1).numpy()[0]
        right_np = (right_np * 255).astype(np.uint8)

        # concat
        frame = np.concatenate([left_np, right_np], axis=1)
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

        video.write(frame)

video.release()

print("Video saved:", save_path)